In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'Scripts'))  # helper modules (run_config, obs, standardplots) live in LPNE/Scripts

from run_config import PROJECT  # project root (LPNE/), found relative to run_config.py - no hard-coded paths
import LPNE
import AREmulator
import run_config as rc
import statschecker as sc
import standardplots as sp
import emusetup_and_mc as emc   # shared emulator build / training and Monte Carlo IC helpers

import numpy as np
import numpy.random as npr
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import r2_score, mean_squared_error
import tensorflow as tf
from tensorflow.keras import layers, Model
import tensorflow.keras.backend as K
import keras as ke
from tensorflow.keras import regularizers
import optuna as opt
seed = 42

In [ ]:
# import data and set up config

# config
DATASET = rc.DATASET        # '1461ka'
INSOL = 'insol_ber90'         # key of rc.STARTS: 'early', 'mid', 'late' or 'default' (80:20 split)
N_FEATURES = rc.NFEATURES

PARAMS = {'conv filters': 30, 'conv kernel': 10, 'dense unit': 28, 
          'dropout rate': 0.3353763108415839, 'learning rate': 0.00038902232764492545, 
          'dwt gamma': 0.6506584806454648, 'dwt beta': 0.4637125670415251, 'dwt eps': 4.1664542731895874e-06, 
          'dwt levels': 2} # taken from optimize_model.ipynb, the 'mid' start optimization set
                           # using the 'mid' start params for comparability (and bc the choice doesn't matter much between sets)

# one config per start time record: the choice is part of the config (and so of the run tag)
cfgs = rc.make_config(DATASET, seed=seed, insol=INSOL, train_start='late', window_kyr = 100, params = PARAMS)

print("run tag :", cfgs['run_tag'])
#print("weights :", cfgs['weights'])
print("params  :", cfgs['params'])
print("train/val start time: ", cfgs['training_start'])
print("train/val frac: ", cfgs['train_frac'])

In [ ]:
# load in and split data
raw_ber90 = rc.load_data(cfgs) 
splits_late = rc.split_data(raw_ber90, cfgs)

print(f'using an late pleistocene start: {rc.describe_split(splits_late)}')

# time splits differ between start times, since the test/train boundary moves
ltestt, ltraint, lvalt = splits_late['test_time'], splits_late['train_time'], splits_late['validation_time']

In [ ]:
# normalize data (each start gets its own scaler, fit on its own training split)
scale_l, train_l, val_l, test_l = rc.fit_and_transform_splits(splits_late)
# can also get and transform standard deviations of records

# chunk normalized data into windows
X_train_l, Y_train_l, Insol_train_l = LPNE.create_windows(train_l, cfgs['window_size'], cfgs['rollout_length'])
X_val_l, Y_val_l, Insol_val_l = LPNE.create_windows(val_l, cfgs['window_size'], cfgs['rollout_length_validation'])

In [ ]:
params_l = cfgs['params']
paths_l = rc.run_paths(cfgs['run_tag'])
emu_l = emc.build_emu(cfgs, params_l)

In [ ]:
print(f"Loading from the late start...")
emu_l.load_weights(str(paths_l['weights']))

In [ ]:
w = cfgs['late']['window_size']

# rollout seeded from the same oldest window, continued through train and validation to the present.
# Uses the full scaled record instead of joining the splits: `+` on numpy arrays adds them elementwise,
# and split_data leaves a 1-row gap between test and train, so joined splits would skip an age
fulll = scale_l.transform(splits_late['full'])
fulltl = splits_late['time']
rollout_full_l = sp.teacherforced_rollout(emu_l, fulll, w)
# boundary between this start's test period and its train/validation period (split_data skips the row in between)
test_end_l = (ltestt.min() + ltraint.max()) / 2
print(test_end_l)